In [285]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np 
import pandas as pd 
import xgboost as xgb
from xgboost.sklearn import XGBClassifier
from sklearn import metrics   #Additional scklearn functions
from sklearn.model_selection import GridSearchCV
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 12, 4

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Loading In Data

In [286]:
education_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_Education_train_set.csv')
education_test = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_Education_test_set.csv')
household_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_HouseholdInfo_train_set.csv')
household_test = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_HouseholdInfo_test_set.csv')
subjective_poverty_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_SubjectivePoverty_train_set.csv')
sample_submission = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/sample_submission.csv')

subjective_poverty_train.head(50)
#household_train.head(50)
#education_train.head(50)

,psu_hh_idcode,subjective_poverty_1,subjective_poverty_2,subjective_poverty_3,subjective_poverty_4,subjective_poverty_5,subjective_poverty_6,subjective_poverty_7,subjective_poverty_8,subjective_poverty_9,subjective_poverty_10
0,30_8_1,0,0,0,1,0,0,0,0,0,0
1,194_1_2,1,0,0,0,0,0,0,0,0,0
2,224_6_1,0,0,1,0,0,0,0,0,0,0
3,323_10_1,0,0,0,0,1,0,0,0,0,0
4,428_10_1,0,0,0,1,0,0,0,0,0,0
5,809_6_2,0,0,1,0,0,0,0,0,0,0
6,138_2_1,0,0,0,0,1,0,0,0,0,0
7,329_1_1,0,1,0,0,0,0,0,0,0,0
8,409_8_1,0,0,0,1,0,0,0,0,0,0
9,186_4_2,0,0,0,0,1,0,0,0,0,0


Preparing Data For Training

In [287]:
subjective_poverty_train[['psu', 'hh', 'idcode']] = subjective_poverty_train['psu_hh_idcode'].str.split('_', expand=True).astype(int)


train_data = pd.merge(education_train, household_train, on=['psu', 'hh', 'idcode'], how='inner')

train_data = pd.merge(train_data, subjective_poverty_train, on=['psu', 'hh', 'idcode'], how='inner')

# Creating a new column 'Genre' based on a condition
train_data['Sub_Pov_Ranking'] = 0  # Initialize the 'Genre' column with 'Other' as the default value

# Using DataFrame.loc[] to set values based on the condition
train_data.loc[train_data['subjective_poverty_1'] == 1, 'Sub_Pov_Ranking'] = 0
train_data.loc[train_data['subjective_poverty_2'] == 1, 'Sub_Pov_Ranking'] = 1
train_data.loc[train_data['subjective_poverty_3'] == 1, 'Sub_Pov_Ranking'] = 2
train_data.loc[train_data['subjective_poverty_4'] == 1, 'Sub_Pov_Ranking'] = 3
train_data.loc[train_data['subjective_poverty_5'] == 1, 'Sub_Pov_Ranking'] = 4
train_data.loc[train_data['subjective_poverty_6'] == 1, 'Sub_Pov_Ranking'] = 5
train_data.loc[train_data['subjective_poverty_7'] == 1, 'Sub_Pov_Ranking'] = 6
train_data.loc[train_data['subjective_poverty_8'] == 1, 'Sub_Pov_Ranking'] = 7
train_data.loc[train_data['subjective_poverty_9'] == 1, 'Sub_Pov_Ranking'] = 8
train_data.loc[train_data['subjective_poverty_10'] == 1, 'Sub_Pov_Ranking'] = 9


5334
   psu  hh  idcode  q01  q02_x  q03_x  q04_x  q05  q06_x  q07_x  ...  \
0  441   2       3    1      1      1    2.0  3.0    3.0    0.0  ...   
1  647   7       1    1      1      1    2.0  2.0    2.0    0.0  ...   
2  756   4       1    1      1      1    2.0  3.0    3.0    0.0  ...   
3   25   4       3    1      1      1    6.0  1.0    3.0    2.0  ...   
4  132   6       3    1      1      1    2.0  3.0    3.0    0.0  ...   

   subjective_poverty_1  subjective_poverty_2  subjective_poverty_3  \
0                     0                     0                     0   
1                     0                     0                     0   
2                     0                     0                     0   
3                     0                     0                     0   
4                     0                     0                     1   

   subjective_poverty_4  subjective_poverty_5  subjective_poverty_6  \
0                     1                     0                   

In [288]:
def modelfit(alg, dtrain, target, useTrainCV=False, cv_folds=5, early_stopping_rounds=50):
    
    if useTrainCV:
        xgb_param = alg.get_xgb_params()
        xgtrain = xgb.DMatrix(dtrain[predictors].values, label=dtrain[target].values)
        cvresult = xgb.cv(xgb_param, xgtrain, num_boost_round=alg.get_params()['n_estimators'], nfold=cv_folds,
            metrics='auc', early_stopping_rounds=early_stopping_rounds)
        alg.set_params(n_estimators=cvresult.shape[0])
    
    #Fit the algorithm on the data
    alg.fit(dtrain, target)
        
    #Predict training set:
    dtrain_predictions = alg.predict(dtrain[predictors])
    dtrain_predprob = alg.predict_proba(dtrain[predictors])[:,1]
        
    #Print model report:
    print("\nModel Report")
    print("Accuracy : %.4g" % metrics.accuracy_score(target.values, dtrain_predictions))
    print("AUC Score (Train): %f" % metrics.roc_auc_score(target, dtrain_predprob))
                    
    feat_imp = pd.Series(alg.booster().get_fscore()).sort_values(ascending=False)
    feat_imp.plot(kind='bar', title='Feature Importances')
    plt.ylabel('Feature Importance Score')

In [289]:
target = 'Sub_Pov_Ranking'

train_data['id'] = np.arange(1, len(train_data)+1)


from sklearn.model_selection import train_test_split

train, test = train_test_split(train_data, test_size=0.2)

print(len(train.index))

print(len(test.index))

IDcol = 'id'

non_predictors = ["psu", "hh", 'idcode', "psu_hh_idcode"]


4267
1067


In [297]:
selected_columns = [column for column in train.columns if column.startswith('subjective_poverty')]

#predictors = train[[important_variables]]
predictors = [x for x in train.columns if x not in ["psu", "hh", 'idcode', "psu_hh_idcode", IDcol, target] and x not in selected_columns]

X_train = train[predictors]
print(X_train)
y_train = train['Sub_Pov_Ranking']

# Create an instance of the XGBClassifier
model = XGBClassifier(objective='multi:softprob', enable_categorical =True, max_depth=3, min_child_weight=1, learning_rate=0.1)

# Fit the model to the training data
fit = model.fit(X_train, y_train)

      q01  q02_x  q03_x  q04_x  q05  q06_x  q07_x  Q08  Q09  Q10  ...  q13  \
3948    1      1      1    5.0  4.0    6.0    0.0  2.0  NaN  NaN  ...  2.0   
1311    1      1      1    1.0  4.0    1.0    0.0  2.0  NaN  NaN  ...  1.0   
2115    1      1      1    1.0  8.0    2.0    1.0  2.0  NaN  NaN  ...  3.0   
3949    1      1      1    2.0  4.0    3.0    0.0  2.0  NaN  NaN  ...  2.0   
1229    1      1      1    1.0  8.0    2.0    0.0  2.0  NaN  NaN  ...  1.0   
...   ...    ...    ...    ...  ...    ...    ...  ...  ...  ...  ...  ...   
1987    1      1      1    6.0  4.0    9.0    0.0  2.0  NaN  NaN  ...  2.0   
4274    1      1      1    3.0  2.0    4.0    0.0  2.0  NaN  NaN  ...  2.0   
5310    1      1      1    5.0  5.0    6.0    0.0  2.0  NaN  NaN  ...  NaN   
1208    2      2      1    1.0  4.0    1.0    0.0  2.0  NaN  NaN  ...  1.0   
2172    1      1      1    1.0  8.0    2.0    0.0  2.0  NaN  NaN  ...  3.0   

      q14   q15   q16  q17  q18  q19  q20   q21   q22  
3948  1

In [298]:
predictors = [x for x in test.columns if x not in ["psu", "hh", 'idcode', "psu_hh_idcode", target, IDcol] and x not in selected_columns]

X_test = test[predictors]

y_test = test[selected_columns]

X_predict = model.predict_proba(X_test)

In [299]:
import math

def multiclass_log_loss(actual,predicted):
    loss = 0 
    i = 1
    while i < len(actual):
        q = 0
        while q < len(actual.iloc[i]):
            y_ij = actual.iloc[i][q]
            log_p_ij = math.log(predicted[i][q])
            loss += y_ij*log_p_ij
            q = q + 1
        i = i + 1
    return(loss/ -len(actual))

In [300]:
print(multiclass_log_loss(y_test, X_predict))

1.8698439386414691


/var/folders/g8/vy9w_fxd6r39lbfd4qykf2j80000gn/T/ipykernel_9894/1365599809.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y_ij = actual.iloc[i][q]


Parameter Tuning

In [295]:
# The scorers can be either one of the predefined metric strings or a scorer
# callable, like the one returned by make_scorer
#scoring = {multiclass_log_loss(,model.predict_proba(X_test))}


param_test1 = {
 'max_depth':range(3,10,2),
 'min_child_weight':range(1,6,2),
 'learning_rate':[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
}

gsearch1 = GridSearchCV(estimator = XGBClassifier( objective='multi:softprob', enable_categorical =True, learning_rate =0.1, n_estimators=140, max_depth=5,
 min_child_weight=1, gamma=0, subsample=0.8, colsample_bytree=0.8,
nthread=4, scale_pos_weight=1, seed=27), 
 param_grid = param_test1, scoring="roc_auc",n_jobs=4, cv=5)
gsearch1.fit(X_train,y_train)
gsearch1.best_params_, gsearch1.best_score_

/Users/callum/miniconda3/envs/condaenv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [18:06:10] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/callum/miniconda3/envs/condaenv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [18:06:10] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/callum/miniconda3/envs/condaenv/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [18:06:10] WARNING: /var/folders/k1/30mswbxs7r1g6zwn8y4fyt500000gp/T/abs_d9k8pmaj4_/croot/xgboost-split_1724073758172/work/src/learner.cc:740: 
Parameters: { "scale_pos_weight" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/

({'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1}, nan)

In [296]:
print(gsearch1.best_params_, gsearch1.best_score_)

{'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1} nan
